In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches_mag10_thre25')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        # cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        # # cell_label = cell_label.apply(lambda x: x*100.0)
        
        # gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        # label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        # label_index_set = set(label_df.index)
        # patch_index_set = set(patch_list)
        # and_set = label_index_set & patch_index_set

        # patch_list = list(and_set)
        # self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches_mag10_thre25', patch_id)
        patch = Image.open(patch_path+'.jpeg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        # label = self.label_df.loc[patch_id].values
        # label = torch.Tensor(label)

        return patch_id, data

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/brca/'
tif_list = glob(tif_list + '/*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-3C-AALK-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-4H-AAAK-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-5L-AAT0-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-5L-AAT1-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-5T-A9QA-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SB-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SD-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SF-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SH-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SI-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SJ-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca/TCGA-A1-A0SK-01Z-00-DX1',
 '/data1/r20user3/shared_project/Hist2Cell/data/brca

In [3]:
len(tif_list)

826

In [4]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['TCGA-3C-AALK-01Z-00-DX1',
 'TCGA-4H-AAAK-01Z-00-DX1',
 'TCGA-5L-AAT0-01Z-00-DX1',
 'TCGA-5L-AAT1-01Z-00-DX1',
 'TCGA-5T-A9QA-01Z-00-DX1',
 'TCGA-A1-A0SB-01Z-00-DX1',
 'TCGA-A1-A0SD-01Z-00-DX1',
 'TCGA-A1-A0SF-01Z-00-DX1',
 'TCGA-A1-A0SH-01Z-00-DX1',
 'TCGA-A1-A0SI-01Z-00-DX1',
 'TCGA-A1-A0SJ-01Z-00-DX1',
 'TCGA-A1-A0SK-01Z-00-DX1',
 'TCGA-A1-A0SM-01Z-00-DX1',
 'TCGA-A1-A0SN-01Z-00-DX1',
 'TCGA-A1-A0SP-01Z-00-DX1',
 'TCGA-A1-A0SQ-01Z-00-DX1',
 'TCGA-A2-A04N-01Z-00-DX1',
 'TCGA-A2-A04P-01Z-00-DX1',
 'TCGA-A2-A04Q-01Z-00-DX1',
 'TCGA-A2-A04R-01Z-00-DX1',
 'TCGA-A2-A04T-01Z-00-DX1',
 'TCGA-A2-A04U-01Z-00-DX1',
 'TCGA-A2-A04V-01Z-00-DX1',
 'TCGA-A2-A04W-01Z-00-DX1',
 'TCGA-A2-A04X-01Z-00-DX1',
 'TCGA-A2-A04Y-01Z-00-DX1',
 'TCGA-A2-A0CK-01Z-00-DX1',
 'TCGA-A2-A0CL-01Z-00-DX1',
 'TCGA-A2-A0CM-01Z-00-DX1',
 'TCGA-A2-A0CO-01Z-00-DX1',
 'TCGA-A2-A0CP-01Z-00-DX1',
 'TCGA-A2-A0CQ-01Z-00-DX1',
 'TCGA-A2-A0CR-01Z-00-DX1',
 'TCGA-A2-A0CS-01Z-00-DX1',
 'TCGA-A2-A0CT-01Z-00-DX1',
 'TCGA-A2-A0CU-01Z-0

In [5]:
# prepare my GPU

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [7]:
from tqdm import tqdm
import torch

batch_size = 512
patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/brca"

# graphs = dict()
for slide in tqdm(test_slides):
    # print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=12)

    spots_coord = pd.read_csv(os.path.join("/data1/r20user3/shared_project/Hist2Cell/data/brca", slide, "spots_mag10_thre25.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            # label = label.to(device)
            # label = label.float()
            # label = label.squeeze()
            # test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    # for i in range(len(test_label_array)):
    #     if len(test_label_array[i].shape) <= 1:
    #         test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    # test_label_array = np.concatenate(test_label_array)
    
    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(item.split('_')[0]), int(item.split('_')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 2 or x2 >= x1 + 2 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    # graphs[slide] = {
    #     'features': test_prediction_array,
    #     # 'labels': test_label_array,
    #     'adj': adj,
    #     'names': test_names,
    #     'coord': spots_coord.loc[test_names].values,
    # }
    
    x = torch.Tensor(test_prediction_array)
    from torch_geometric.utils import dense_to_sparse
    adj = torch.Tensor(adj)
    edge_index, _ = dense_to_sparse(adj)
    from torch_geometric.data import Data
    pos = torch.Tensor(spots_coord.loc[test_names].values)

    data = Data(x=x, edge_index=edge_index, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca_mag10_thre25", slide+".pt"))

    # print("OK")

  0%|          | 0/826 [00:00<?, ?it/s]/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 826/826 [11:22:33<00:00, 49.58s/it]   


In [51]:
from glob import glob
import torch


pt_list = glob("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/*.pt")
for pt in pt_list:
    data = torch.load(pt)
    print(pt)
    print(data)

/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-UU-A93S-01Z-00-DX1.pt
Data(x=[8056, 3, 224, 224], edge_index=[2, 70288], pos=[8056, 2])
/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-D8-A27F-01Z-00-DX1.pt
Data(x=[4811, 3, 224, 224], edge_index=[2, 42025], pos=[4811, 2])
/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-BH-A0HU-01Z-00-DX1.pt
Data(x=[2345, 3, 224, 224], edge_index=[2, 18955], pos=[2345, 2])
/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A2-A0T3-01Z-00-DX1.pt
Data(x=[35, 3, 224, 224], edge_index=[2, 75], pos=[35, 2])
/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A2-A4RX-01Z-00-DX1.pt
Data(x=[2205, 3, 224, 224], edge_index=[2, 17279], pos=[2205, 2])
/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-JL-A3YW-01Z-00-DX1.pt
Data(x=[654, 3, 2

In [32]:
graphs['TCGA-A2-A0CS-01Z-00-DX1']['features'].shape

(2754, 3, 224, 224)

In [33]:
graphs['TCGA-A2-A0CS-01Z-00-DX1']['coord'].shape

(2754, 2)

In [35]:
graphs['TCGA-A2-A0CS-01Z-00-DX1']['coord']

array([[67, 65],
       [23, 68],
       [13, 29],
       ...,
       [30, 71],
       [73, 56],
       [59, 14]])

In [34]:
graphs['TCGA-A2-A0CS-01Z-00-DX1']['names']

['67_65',
 '23_68',
 '13_29',
 '71_41',
 '37_22',
 '12_25',
 '70_71',
 '71_50',
 '23_27',
 '68_62',
 '68_68',
 '31_69',
 '17_31',
 '16_48',
 '63_31',
 '67_68',
 '33_9',
 '20_11',
 '66_58',
 '37_23',
 '35_34',
 '28_68',
 '13_23',
 '65_23',
 '36_33',
 '9_66',
 '16_33',
 '68_35',
 '69_67',
 '15_24',
 '70_22',
 '21_40',
 '39_25',
 '71_73',
 '71_26',
 '17_10',
 '9_58',
 '35_40',
 '17_32',
 '48_39',
 '19_69',
 '39_48',
 '16_69',
 '25_23',
 '55_16',
 '56_57',
 '64_23',
 '65_38',
 '62_44',
 '70_69',
 '8_42',
 '34_37',
 '60_20',
 '74_54',
 '16_40',
 '66_52',
 '25_59',
 '53_45',
 '25_58',
 '62_52',
 '67_49',
 '31_50',
 '26_31',
 '13_9',
 '36_64',
 '24_61',
 '31_61',
 '37_15',
 '31_18',
 '61_66',
 '11_62',
 '34_16',
 '36_58',
 '51_16',
 '70_36',
 '21_37',
 '38_65',
 '42_16',
 '26_27',
 '28_59',
 '34_22',
 '71_36',
 '57_49',
 '69_36',
 '16_38',
 '64_34',
 '75_72',
 '9_34',
 '30_19',
 '29_20',
 '67_30',
 '34_27',
 '56_15',
 '30_68',
 '27_18',
 '34_23',
 '33_22',
 '8_15',
 '22_68',
 '35_20',
 '30_63

In [36]:
graph = graphs
for slide in graph:
    print(slide)

TCGA-A2-A0CS-01Z-00-DX1


In [37]:
from torch import Tensor
import torch
import os
from tqdm import tqdm

for slide_name in tqdm(graph):
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca", slide_name+".pt"))

In [47]:
import torch

data1 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A2-A0CS-01Z-00-DX1.pt")
data2 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A2-A0CS-01Z-00-DX1.pt")

In [48]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
print(data)

DataBatch(x=[5508, 3, 224, 224], edge_index=[2, 43292], pos=[5508, 2], batch=[5508], ptr=[3])


In [40]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

from torch_geometric.seed import seed_everything
seed_everything(0)

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

i = 0
for sampled_data in loader:
    print(sampled_data.input_id)
    i = i + 1
    if i > 10:
        break

tensor([1542])
tensor([4424])
tensor([1503])
tensor([1613])
tensor([4777])
tensor([1684])
tensor([981])
tensor([4646])
tensor([428])
tensor([4954])
tensor([3079])


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


In [41]:
sampled_data

DataBatch(x=[18, 3, 224, 224], edge_index=[2, 114], pos=[18, 2], ptr=[3], n_id=[18], e_id=[114], input_id=[1], batch_size=1)

In [42]:
sampled_data.pos

tensor([[26., 15.],
        [25., 14.],
        [27., 16.],
        [25., 16.],
        [27., 15.],
        [26., 14.],
        [27., 14.],
        [25., 15.],
        [26., 16.],
        [25., 13.],
        [24., 14.],
        [26., 13.],
        [24., 13.],
        [24., 15.],
        [28., 16.],
        [28., 15.],
        [25., 17.],
        [24., 16.]])